# §37 — State katkısı biriken geçmişle nasıl değişiyor?

**Otorite ön-kayıt `RESULTS.md` §37'de.** Bu markdown kolaylık kopyasıdır.

§36 katkıyı **tek bir mesafede** ölçtü: exp +0.0350, cubic −0.2817 nat/token.
Ölçmediği şey, katkının biriken geçmişle nasıl davrandığı.

**Δ(D) = CE(state ölçüm chunk'ında sıfır) − CE(state D chunk geriden taşınıyor)**
D ∈ {1, 2, 4, 8, 16}

**Neden Faz A'dan önce:** cevap Faz A'nın hangi problemi çözeceğini belirliyor.
- **DOYUYOR** → kanal dolu, kısıt **kapasite**; seçici yazım yanlış kaldıraç.
- **BÜYÜYOR** → kanal kararlı ve az kullanılmış, kısıt **ne yazıldığı**;
  seçici yazım doğru kaldıraç.
- **AZALIYOR** → birikim exp için de zararlı; "daha iyi içerik yaz" yönü kapanır.

**D=1 iç tutarlılık kontrolüdür:** §36'nın koşulunu tekrar üretir ve
+0.0350 / −0.2817 civarına düşmelidir. Düşmezse iki koşu arasında bir şey
farklı demektir ve tarama kıyaslanabilir değildir.

**Test önceden Wilcoxon.** Eşik 0.02 nat/token. Güç kapısı: exp'te Δ(1) > 0,
p < 0.05 — yoksa menzili ölçülecek bir katkı yok, **sonuçsuz** yazılır.

**Ölçmediği:** Δ(D), *D chunk geçmişe sahip olmanın* değeri; *tam D chunk
önceki içeriğin* değeri değil. Menzil ve birikim farklı büyüklükler; bu deney
birikimi ölçüyor.


In [ ]:
# ================= KIMLIK =================
print("NOTEBOOK: content_vs_priming_v38.ipynb  |  RESULTS.md §38")
print("SORU: state'in katkisi ICERIK mi, alan/uslup isinmasi mi?")
print("CIKTI: v38_raw.json  (Kaggle: /kaggle/working)")
print("=" * 42)
# ==========================================
# --- 1. KURULUM ---
import os, sys, glob, json, subprocess, math, time
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'GPU YOK. Accelerator > T4 sec.'
OUT = os.path.join(BASE,'v37'); os.makedirs(OUT, exist_ok=True)
print('repo:', REPO, '| cihaz:', torch.cuda.get_device_name(0), '| cikti:', OUT)

In [ ]:
# --- 2. IKIZ CHECKPOINTLERI BUL + PARMAK IZIYLE DOGRULA + KATMAN KUMESI ---
# NOT: checkpoint'ler .gitignore'da (*.pt, checkpoints/) -> repo klonunda YOKLAR.
# Kaggle: Add Data > Your Datasets ile iki .pt dosyasini yukle (bkz. markdown).
#
# DOSYA ADINA GUVENILMEZ. KAYNAK.md'deki ders: Run 1'in 'hfp_graft_final.pt'si
# bir ara 'run5' etiketiyle dolasti. Kimlik PARMAK IZIYLE dogrulanir:
#   Run 1  -> out_gain ort ~0.75   (YANLIS dosya)
#   Run 5  -> out_gain ort ~0.237  (cubic ikizi)
#   Run 6  -> out_gain ort ~0.239  (exp ikizi)
import re
ROOTS = ['/kaggle/input', REPO, BASE, '/content/drive/MyDrive', '/content', os.path.expanduser('~')]
cands = []
for r in ROOTS:
    if r and os.path.isdir(r): cands += glob.glob(f'{r}/**/*.pt', recursive=True)
cands = sorted(set(cands))
assert cands, ('HIC .pt DOSYASI YOK.\n'
               '  Kaggle: Add Data > Upload > iki dosyayi yukle:\n'
               '    checkpoints/graft_run5/hfp_graft_final.pt      (cubic)\n'
               '    checkpoints/graft_run6_exp/hfp_graft_exp_final.pt (exp)\n'
               '  Bunlar .gitignore\'da oldugu icin repo klonunda YOK.')

def probe(path):
    sd = torch.load(path, map_location='cpu')
    if isinstance(sd, dict) and 'm' in sd and isinstance(sd['m'], dict): sd = sd['m']
    og = [v.flatten() for k, v in sd.items() if k.endswith('out_gain')]
    if not og: return None
    og = torch.cat(og)
    ls = sorted({int(m.group(1)) for k in sd
                 for m in [re.search(r'layers\.(\d+)\.self_attn', k)] if m})
    return {'sd': sd, 'n': len(sd), 'og': float(og.mean()), 'ogsd': float(og.std()), 'layers': ls}

print(f"{'out_gain':>9} {'std':>7} {'tensor':>7} {'kat':>4}  dosya")
print('-'*84)
info = {}
for c in cands:
    p = probe(c)
    if p is None: continue
    info[c] = p
    flag = '  <-- Run1? (out_gain ~0.75, YANLIS)' if p['og'] > 0.5 else ''
    print(f"{p['og']:>9.4f} {p['ogsd']:>7.4f} {p['n']:>7} {len(p['layers']):>4}  {c}{flag}")
assert info, 'Bulunan .pt dosyalarinin hicbiri graft checkpointi degil (out_gain yok).'

# Secim: exp = dosya adinda 'exp'; cubic = 'exp' YOK. Ikisinde de parmak izi kapisi.
def pick(arm):
    want_exp = (arm == 'exp')
    hits = [c for c, p in info.items()
            if (('exp' in os.path.basename(c).lower()) == want_exp)
            and 'final' in os.path.basename(c).lower()
            and 0.15 <= p['og'] <= 0.40]          # PARMAK IZI KAPISI: Run1'i (~0.75) eler
    assert hits, (f'{arm} ikizi bulunamadi. Ya dosya yok, ya parmak izi disinda '
                  f'(0.15-0.40 bandi). Yukaridaki tabloya bak: Run1 (~0.75) KABUL EDILMEZ.')
    assert len(hits) == 1, f'{arm} icin BIRDEN FAZLA aday: {hits}. Fazlasini kaldir.'
    return hits[0]

CKPT = {a: pick(a) for a in ('cubic', 'exp')}
SD_CUBIC, SD_EXP = info[CKPT['cubic']]['sd'], info[CKPT['exp']]['sd']
L_CUBIC, L_EXP   = info[CKPT['cubic']]['layers'], info[CKPT['exp']]['layers']
print()
for a in ('cubic', 'exp'):
    i = info[CKPT[a]]
    print(f'{a:>6}: out_gain {i["og"]:.4f} | tensor {i["n"]} | katman {len(i["layers"])} | {CKPT[a]}')
assert L_CUBIC == L_EXP, ('IKIZ DEGILLER: katman kumeleri farkli -> tek-degisken '
                          f'kosulu ihlal, deney KOSULAMAZ.\n  cubic {L_CUBIC}\n  exp   {L_EXP}')
GRAFT_LAYERS = L_CUBIC
print(f'\nortak katman kumesi ({len(GRAFT_LAYERS)}): {GRAFT_LAYERS}')
print('KAYNAK.md referansi: Run5 out_gain ~0.237 (cubic), Run6 ~0.239 (exp)')


In [ ]:
# --- 3. BASE MODEL + GRAFT KURUCU (yukleme DOGRULAMASI zorunlu, §30 hucre 7) ---
import glob, os
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode,
                                 enable_streaming, reset_streaming, HFPGraftAttention)
ROOTS = ['/kaggle/input', REPO, BASE, os.path.expanduser('~'), '.']
def find_one(pat):
    hits = []
    for r in ROOTS:
        if r and os.path.isdir(r): hits += glob.glob(f'{r}/**/{pat}', recursive=True)
    return max(set(hits), key=os.path.getmtime) if hits else None

# Qwen: config.json ILE AGIRLIGIN AYNI KLASORDE oldugu yeri ara.
# Not: repodaki hf_upload/hf_release/config.json agirlik icermiyor; sadece
# 'config.json' aramak onu bulup SRC'yi bos birakiyordu.
_cands = []
for _r in ROOTS:
    if not (_r and os.path.isdir(_r)): continue
    for _cfg in glob.glob(f'{_r}/**/config.json', recursive=True):
        _d = os.path.dirname(_cfg)
        if glob.glob(f'{_d}/*.safetensors') or glob.glob(f'{_d}/*.bin'):
            _cands.append(_d)
assert _cands, ('QWEN YOK. Kaggle sag panel > Add Input > Models > Qwen2.5-1.5B.\n'
                '  (Datasets sekmesi degil, Models sekmesi.)')
SRC = _cands[0]
tok = AutoTokenizer.from_pretrained(SRC)
print('base model:', SRC)


def build(arm):
    """arm in {'cubic','exp'} -> yuklenmis, DOGRULANMIS grafted model."""
    mode = 'cubic_flux_chunked' if arm == 'cubic' else 'exp'
    m = AutoModelForCausalLM.from_pretrained(SRC, torch_dtype=torch.float32).to(DEV).eval()
    graft_llama(m, GraftConfig(decay_mode=mode, write_rule='hybrid',
                               key_feature_map='dpfp', rec_block=16), layers=GRAFT_LAYERS)
    for mm in m.modules():
        if isinstance(mm, HFPGraftAttention): mm.out_gain.data.fill_(0.1)
    sd = SD_CUBIC if arm == 'cubic' else SD_EXP
    m.load_state_dict(sd, strict=False)
    set_graft_mode(m, 'student'); m.config.use_cache = True
    # --- YUKLEME DOGRULAMASI (strict=False sessizce hicbir sey yuklemeyebilir) ---
    own = dict(m.state_dict())
    matched = [k for k in sd if k in own and own[k].shape == sd[k].shape]
    missing = [k for k in sd if k not in own]
    same = sum(1 for k in matched if torch.equal(own[k].to(DEV), sd[k].to(DEV)))
    og = torch.cat([mm.out_gain.detach().flatten() for mm in m.modules()
                    if isinstance(mm, HFPGraftAttention)])
    ok = (len(matched) == len(sd)) and (same == len(matched)) and (og.std() > 1e-4)
    print(f'[{arm}] tensor {len(sd)} | eslesen {len(matched)} | bit-bit ayni {same} | '
          f'eksik isim {len(missing)} | out_gain ort {og.mean():.4f} std {og.std():.4f}')
    assert ok, (f'[{arm}] CHECKPOINT YUKLENMEDI. out_gain std ~0 ve ort ~0.1 ise '
                f'agirliklar EGITIMSIZ -> hicbir sayi raporlanamaz. eksik: {missing[:3]}')
    print(f'[{arm}] YUKLEME DOGRULANDI')
    return m
print('\nkurucu hazir (build("cubic") / build("exp"))')

In [ ]:
# --- 4. DEGERLENDIRME METNI (held-out) ---
# S1 distilasyonu WikiText-103 TRAIN uzerinde yapildi -> VALIDATION held-out'tur.
CHUNK, HEAD, N_CHUNK = 256, 32, 120   # chunk uzunlugu | endpoint penceresi | chunk sayisi
try:
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='validation')
    text = '\n\n'.join(t for t in ds['text'] if len(t.strip()) > 200)
    kaynak = 'wikitext-103-raw-v1/validation'
except Exception as e:
    p = find_one('wiki.valid.tokens') or find_one('wiki.valid.raw')
    assert p, f'Degerlendirme metni yok ({type(e).__name__}). WikiText-103 valid ekle.'
    text = open(p, encoding='utf-8').read(); kaynak = p
ids = tok(text, return_tensors='pt').input_ids[0]
need = CHUNK * (N_CHUNK + 1)
assert ids.numel() >= need, f'metin kisa: {ids.numel()} < {need}'
ids = ids[:need]
print(f'kaynak: {kaynak}\ntoken: {ids.numel():,} | chunk {CHUNK} x {N_CHUNK+1} | endpoint = ilk {HEAD} token')

In [ ]:
# --- 5. KOSU (§38): D in {1,4,16} x uc kosul (bitisik / rastgele / sifir) ---
# ON-KAYIT: RESULTS.md §38. Ortak olcum penceresi ONCEDEN sabit: t = TMIN..TMAX.
# §37'nin tasarim kusuru (her D'nin farkli chunk kumesinde olcmesi) burada YOK.
import torch.nn.functional as F
import random

DS          = [1, 4, 16]
TMIN, TMAX  = 16, N_CHUNK      # t = 16..120 -> n = 105, HER D ve HER kosul icin ayni
SEED        = 20260803         # rastgele gecmis secimi tekrarlanabilir olsun

def _rand_hist(D, t, rng):
    """Ayni akistan D chunk sec; t etrafindaki +-D pencereyi DISLA.
    Amac: gecmis miktari/alan/uslup sabit, YALNIZCA 'devam-ilgili baglam'
    ozelligi kaldirilmis olsun."""
    yasak = set(range(max(0, t - D), min(N_CHUNK + 1, t + D + 1)))
    havuz = [c for c in range(N_CHUNK) if c not in yasak]
    assert len(havuz) >= D, f'havuz kucuk: {len(havuz)} < {D} (t={t})'
    return rng.sample(havuz, D)

@torch.no_grad()
def feed(m, chunks):
    for c in chunks:
        xf = ids[c*CHUNK:(c+1)*CHUNK].unsqueeze(0).to(DEV)
        m(xf, past_key_values=DynamicCache(), use_cache=True)

@torch.no_grad()
def score(m, t):
    x  = ids[t*CHUNK:(t+1)*CHUNK].unsqueeze(0).to(DEV)
    lg = m(x, past_key_values=DynamicCache(), use_cache=True).logits
    return F.cross_entropy(lg[0, :HEAD, :], x[0, 1:HEAD+1]).item()

@torch.no_grad()
def run_D(m, D):
    """Cache HER chunk'ta sifir (DynamicCache()); tek kanal recurrent state."""
    enable_streaming(m, True)
    rng = random.Random(SEED + D)
    o = {'contig': [], 'random': [], 'reset': [], 't': [], 'rand_idx': []}
    for t in range(TMIN, TMAX + 1):
        assert t - D >= 0, f'bitisik gecmis tasiyor: t={t}, D={D}'
        # 1) BITISIK: t'den hemen once gelen D chunk  (= §37'nin kosulu)
        reset_streaming(m); feed(m, range(t - D, t))
        o['contig'].append(score(m, t))
        # 2) RASTGELE: ayni akistan D chunk, t civari haric
        ridx = _rand_hist(D, t, rng)
        reset_streaming(m); feed(m, ridx)
        o['random'].append(score(m, t))
        # 3) SIFIR: state yok
        reset_streaming(m)
        o['reset'].append(score(m, t))
        o['t'].append(t); o['rand_idx'].append(ridx)
    return o

RES = {}
for arm in ('cubic', 'exp'):
    t0 = time.time(); m = build(arm); RES[arm] = {}
    for D in DS:
        o = run_D(m, D); RES[arm][D] = o
        dc = [r - c for r, c in zip(o['reset'], o['contig'])]
        dr = [r - c for r, c in zip(o['reset'], o['random'])]
        gp = [a - b for a, b in zip(dc, dr)]
        print(f'[{arm}] D={D:>2}  D_bitisik={sum(dc)/len(dc):+.4f}  '
              f'D_rastgele={sum(dr)/len(dr):+.4f}  Gap={sum(gp)/len(gp):+.4f}  '
              f'(n={len(dc)})', flush=True)
    del m; torch.cuda.empty_cache()
    print(f'[{arm}] bitti ({time.time()-t0:.0f}s)\n')

json.dump({'chunk': CHUNK, 'head': HEAD, 'n_chunk': N_CHUNK, 'DS': DS,
           'tmin': TMIN, 'tmax': TMAX, 'seed': SEED, 'kaynak': kaynak,
           'layers': GRAFT_LAYERS, 'res': RES},
          open(f'{OUT}/v38_raw.json', 'w'), indent=2)
print('ham veri:', f'{OUT}/v38_raw.json')


In [ ]:
# --- 6. GUC KAPISI + ON-KAYITLI HUKUM (§38) ---
import statistics as st, math

def wilcoxon(d):
    dd = [x for x in d if x != 0]; n = len(dd)
    if n < 6: return float('nan'), float('nan')
    order = sorted(range(n), key=lambda i: abs(dd[i])); rank = [0.0]*n
    i = 0
    while i < n:
        j = i
        while j+1 < n and abs(dd[order[j+1]]) == abs(dd[order[i]]): j += 1
        avg = (i+j)/2 + 1
        for k in range(i, j+1): rank[order[k]] = avg
        i = j + 1
    W = sum(rank[i] for i in range(n) if dd[i] > 0)
    z = (W - n*(n+1)/4) / math.sqrt(n*(n+1)*(2*n+1)/24)
    return z, math.erfc(abs(z)/math.sqrt(2))

DC, DR, GAP = {}, {}, {}
for a in ('cubic', 'exp'):
    DC[a], DR[a], GAP[a] = {}, {}, {}
    for D in DS:
        o = RES[a][D]
        DC[a][D] = [r - c for r, c in zip(o['reset'], o['contig'])]
        DR[a][D] = [r - c for r, c in zip(o['reset'], o['random'])]
        GAP[a][D] = [x - y for x, y in zip(DC[a][D], DR[a][D])]

# --- eslestirme dogrulamasi: tum kollar ayni t vektorunde mi ---
tset = {tuple(RES[a][D]['t']) for a in ('cubic','exp') for D in DS}
assert len(tset) == 1, 'KOLLAR AYNI OLCUM PENCERESINDE DEGIL -- hukum verme'
print(f'olcum penceresi: t = {min(tset.pop())}..{TMAX}, n = {len(DC["exp"][DS[0]])}  [tum kollar ayni]')

print('\n=== §38 tablosu (nat/token; POZITIF = state yardim ediyor) ===')
for a in ('exp', 'cubic'):
    print(f'\n  --- {a} ---')
    print(f"  {'D':>3} {'D_bitisik':>22} {'D_rastgele':>22} {'Gap = icerige ozgu':>26}")
    for D in DS:
        r = ''
        for arr in (DC[a][D], DR[a][D], GAP[a][D]):
            m_ = st.mean(arr); _, p = wilcoxon(arr)
            r += f'  {m_:+.4f} (p={p:.3g})'.rjust(24 if arr is not GAP[a][D] else 26)
        print(f'  {D:>3}{r}')

print('\n=== IC TUTARLILIK: exp D=1 bitisik, §37 hizali degeriyle ===')
m1 = st.mean(DC['exp'][1]); _, p1 = wilcoxon(DC['exp'][1])
print(f'  bu kosu = {m1:+.4f}   §37 (t=16..120 hizali) = +0.1811')
print(f'  fark = {m1 - 0.1811:+.4f}  (buyukse kosular KIYASLANAMAZ -- once bunu coz)')

GATE = (m1 > 0) and (p1 < 0.05)
print(f'\n=== GUC KAPISI: exp D=1 bitisik = {m1:+.4f}, p={p1:.4g} -> '
      f'{"GECILDI" if GATE else "GECILMEDI"} ===')

print('\n=== ON-KAYITLI HUKUM (§38; birincil kol=exp, birincil mesafe D=16) ===')
if not GATE:
    print('  SONUCSUZ (null DEGIL). En kisa mesafede bile olculebilir katki yok;')
    print('  icerik/isinma ayrimi yapilacak bir sey kalmiyor. Faz A bu girdi')
    print('  olmadan ilerler.')
else:
    g = GAP['exp'][16]; mg = st.mean(g); _, pg = wilcoxon(g)
    dr16 = st.mean(DR['exp'][16])
    print(f'  Gap(16) = {mg:+.4f} nat/token  (n={len(g)}, Wilcoxon p={pg:.4g})')
    print(f'  esikler: ICERIK >= 0.02 (p<0.05)  |  ISINMA <= 0.005')
    print(f'  referans: D_rastgele(16) = {dr16:+.4f}  <- gorunur etkinin alan payi')
    if mg >= 0.02 and pg < 0.05:
        print('\n  => ICERIK TASIYOR: plato gercek, chunk\'a ozgu hafiza.')
        print('     Faz A\'nin kapasite cercevesi DURUR; state buyutme (dpfp_nu)')
        print('     artik gerekcelendirilmis siradaki deney.')
    elif mg <= 0.005:
        print('\n  => ISINMA BASKIN: plato icerige ozgu DEGIL. exp state\'i hafiza')
        print('     degil, alan/uslup adaptoru gibi calisiyor. Faz A mimari isten')
        print('     ONCE yeniden cerceveleniyor; §37\'nin platosu belge boyunca')
        print('     yeniden tanimlanmali.')
    else:
        print('\n  => SONUCSUZ (null DEGIL). Gap 0.005-0.02 arasinda ya da p>=0.05.')
        print('     Hukum YOK, hicbir iddia revize EDILMEZ.')

print('\n--- cubic kolu (HUKUM YOK; §36/§37 kapatti, betimleyici) ---')
for D in DS:
    print(f'  D={D:>2}: bitisik {st.mean(DC["cubic"][D]):+.4f}  '
          f'rastgele {st.mean(DR["cubic"][D]):+.4f}  Gap {st.mean(GAP["cubic"][D]):+.4f}')
